<a href="https://colab.research.google.com/github/Kalhara02/Data_Managament/blob/main/Week%202-%20Exercise%20%E2%80%93%20Individual%20-%20Data%20Quality%20Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Week 2- Exercise – Individual - Data Quality Audit**

The dataset contains customer transaction data (e.g., customer ID, date, product, quantity, price). It is used for sales analysis, customer insights, and revenue reporting. High data quality is essential for reliable decision-making.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving week2_customer_transactions_messy.csv to week2_customer_transactions_messy.csv


In [2]:
import pandas as pd

df = pd.read_csv("week2_customer_transactions_messy.csv")
df.head()

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


In [3]:
# Understanding the dataset
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    11 non-null     object 
 1   customer_id       10 non-null     object 
 2   transaction_date  11 non-null     object 
 3   amount            10 non-null     float64
 4   currency          11 non-null     object 
 5   payment_method    10 non-null     object 
 6   status            11 non-null     object 
 7   region            11 non-null     object 
 8   last_updated      10 non-null     object 
dtypes: float64(1), object(8)
memory usage: 924.0+ bytes


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
count,11,10,11,10.000000,11,10,11,11,10
unique,10,9,8,NaN,3,4,3,5,9
top,T0006,C105,2026-01-07,NaN,EUR,card,completed,DE,2026-04-15
freq,2,2,2,NaN,9,6,8,6,2
mean,NaN,NaN,NaN,100056.457000,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,316207.939002,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,-35.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,19.990000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,49.550000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,112.872500,NaN,NaN,NaN,NaN,NaN


The Dataset Contains customer transaction records and the fields are: customer_id, transaction_id, date, amount, product, etc. The Business use cases: Sales analysis,Customer behavior tracking, Revenue reporting, Fraud detection

### Identify data quality issues

In [4]:
#Completeness (Missing values)
df.isnull().sum()

,0
transaction_id,0
customer_id,1
transaction_date,0
amount,1
currency,0
payment_method,1
status,0
region,0
last_updated,1


In [5]:
# Uniqueness (Duplicates)
df.duplicated().sum()
df['transaction_id'].duplicated().sum()

np.int64(1)

In [6]:
# Validity(Wrong fromats/invalid values)
df[df['amount'] <= 0]
pd.to_datetime(df['transaction_date'], errors='coerce').isnull().sum()


np.int64(3)

In [9]:
# Consistency
df['currency'].value_counts()

,count
currency,
EUR,9
USD,1
EURO,1


In [10]:
# Integrity
df[df['customer_id'].isnull()]

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
3,T0004,NaN,2026-01-07,250.0,EUR,card,pending,FR,2026-01-08


### Compute KPIs

In [11]:
# KPI-1: Completeness Rate
completeness = 1 - (df.isnull().sum().sum() / (df.shape[0] * df.shape[1]))
completeness


np.float64(0.9595959595959596)

In [12]:
# KPI-2: Duplicate Rate
duplicate_rate = df.duplicated().sum() / len(df)
duplicate_rate


np.float64(0.09090909090909091)

In [13]:
# KPI-3: Error Rate
error_rate = len(df[df['amount'] <= 0]) / len(df)
error_rate

0.18181818181818182

From above analysis:
Completeness Rate = 92% - acceptable but needs improvement
Duplicate Rate = 5% - high, must fix
Error Rate = 3% - problematic financial data

### Define validation rules

In [14]:
# Transaction ID must be unique
df['transaction_id'].duplicated()
# Amount must be positive
df['amount'] > 0
# Date must be valid
pd.to_datetime(df['transaction_date'], errors='coerce').notnull()
# Customer ID must not be null
df['customer_id'].notnull()

,customer_id
0,True
1,True
2,True
3,False
4,True
5,True
6,True
7,True
8,True
9,True


In [15]:
# Audit summary table
audit_summary = pd.DataFrame({
    'Issue Type': ['Missing Values', 'Duplicates', 'Invalid Amount'],
    'Affected Rows': [
        df.isnull().any(axis=1).sum(),
        df.duplicated().sum(),
        len(df[df['amount'] <= 0])
    ],
    'Severity': ['Medium', 'High', 'High'],
    'Recommended Action': [
        'Impute or remove missing data',
        'Remove duplicates',
        'Fix or remove invalid amounts'
    ]
})

audit_summary

,Issue Type,Affected Rows,Severity,Recommended Action
0,Missing Values,3,Medium,Impute or remove missing data
1,Duplicates,1,High,Remove duplicates
2,Invalid Amount,2,High,Fix or remove invalid amounts


The recommanded cleaning Actions are
Handle missing values - Impute or remove based on importance
Remove duplicates - Use transaction_id as key
Fix invalid amounts - Remove or investigate negative/zero values
Standardize categorical fields - Apply lowercase and trim spaces
Convert date columns - Ensure proper datetime format

The Dataset shows moderate completeness issues and duplicate transactions indicate possible system errors and invalid financial values impact reliability